# FK/PK Constraint Tests for Wheelie Gold Layer

This notebook validates that PK and FK constraints exist in Delta table metadata.


In [ ]:
# ==============================================================================
# FK/PK CONSTRAINT METADATA TESTS
# ==============================================================================
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("fk_pk_constraint_tests")


def get_create_stmt(full_name: str) -> str:
    """Return the SHOW CREATE TABLE statement as a single string."""
    rows = spark.sql(f"SHOW CREATE TABLE {full_name}").collect()
    if not rows:
        raise AssertionError(f"SHOW CREATE TABLE returned no rows for {full_name}")
    return "\n".join([row[0] for row in rows])


def assert_constraint(full_name: str, constraint_name: str, constraint_type: str):
    """Assert that a constraint exists in table metadata."""
    stmt = get_create_stmt(full_name).lower()
    name = constraint_name.lower()
    type_token = constraint_type.lower()

    if name not in stmt:
        logger.error(f"Missing constraint {constraint_name} on {full_name}")

    assert name in stmt, f"{full_name}: missing constraint {constraint_name}"
    assert type_token in stmt, f"{full_name}: missing {constraint_type} keyword"


constraints = [
    # Primary keys
    ("dim_date", "pk_dim_date", "PRIMARY KEY"),
    ("dim_service_date", "pk_dim_service_date", "PRIMARY KEY"),
    ("dim_rental_date", "pk_dim_rental_date", "PRIMARY KEY"),
    ("dim_return_date", "pk_dim_return_date", "PRIMARY KEY"),
    ("dim_payment_date", "pk_dim_payment_date", "PRIMARY KEY"),
    ("dim_payment_deadline_date", "pk_dim_payment_deadline_date", "PRIMARY KEY"),
    ("dim_staff", "pk_dim_staff", "PRIMARY KEY"),
    ("dim_manager", "pk_dim_manager", "PRIMARY KEY"),
    ("dim_store", "pk_dim_store", "PRIMARY KEY"),
    ("dim_car", "pk_dim_car", "PRIMARY KEY"),
    ("dim_customer", "pk_dim_customer", "PRIMARY KEY"),
    ("dim_equipment", "pk_dim_equipment", "PRIMARY KEY"),
    ("fact_service", "pk_fact_service", "PRIMARY KEY"),
    ("fact_rental", "pk_fact_rental", "PRIMARY KEY"),
    ("bridge_staff_hierarchy", "pk_bridge_staff_hierarchy", "PRIMARY KEY"),
    ("bridge_car_equipment", "pk_bridge_car_equipment", "PRIMARY KEY"),
    ("bridge_equipment_group_equipment", "pk_bridge_equipment_group_equipment", "PRIMARY KEY"),
    # Foreign keys
    ("fact_service", "fk_fact_service_car", "FOREIGN KEY"),
    ("fact_rental", "fk_fact_rental_customer", "FOREIGN KEY"),
    ("fact_rental", "fk_fact_rental_car", "FOREIGN KEY"),
    ("fact_rental", "fk_fact_rental_staff", "FOREIGN KEY"),
    ("fact_rental", "fk_fact_rental_store", "FOREIGN KEY"),
    ("fact_rental", "fk_fact_rental_rental_date", "FOREIGN KEY"),
    ("fact_rental", "fk_fact_rental_return_date", "FOREIGN KEY"),
    ("fact_rental", "fk_fact_rental_payment_date", "FOREIGN KEY"),
    ("fact_rental", "fk_fact_rental_payment_deadline_date", "FOREIGN KEY"),
    ("bridge_staff_hierarchy", "fk_bridge_staff_staff", "FOREIGN KEY"),
    ("bridge_staff_hierarchy", "fk_bridge_staff_manager", "FOREIGN KEY"),
    ("bridge_car_equipment", "fk_bridge_car_equipment_car", "FOREIGN KEY"),
    ("bridge_equipment_group_equipment", "fk_bridge_equipment_group_equipment_equipment", "FOREIGN KEY"),
]

logger.info("=" * 70)
logger.info("EXECUTING FK/PK CONSTRAINT METADATA TESTS")
logger.info("=" * 70)

passed = 0
failed = 0
failed_tests = []

for table_name, constraint_name, constraint_type in constraints:
    full_name = f"wheelie.gold.{table_name}"
    try:
        logger.info(f"Checking {constraint_type}: {full_name}.{constraint_name}")
        assert_constraint(full_name, constraint_name, constraint_type)
        passed += 1
        logger.info(f"✅ PASS: {constraint_name}\n")
    except AssertionError as e:
        failed += 1
        failed_tests.append({
            "table": full_name,
            "constraint": constraint_name,
            "type": constraint_type,
            "error": str(e),
        })
        logger.error(f"❌ FAIL: {constraint_name}")
        logger.error(f"   {str(e)}\n")
    except Exception as e:
        failed += 1
        failed_tests.append({
            "table": full_name,
            "constraint": constraint_name,
            "type": constraint_type,
            "error": f"Unexpected error: {str(e)}",
        })
        logger.error(f"❌ ERROR: {constraint_name}")
        logger.error(f"   Unexpected error: {str(e)}\n")

total = len(constraints)

logger.info("=" * 70)
logger.info("TEST EXECUTION SUMMARY")
logger.info("=" * 70)
logger.info(f"Total Tests: {total}")
logger.info(f"Passed: {passed} ✅")
logger.info(f"Failed: {failed} ❌")
logger.info(f"Success Rate: {(passed/total*100):.1f}%")

if failed > 0:
    logger.error("\n" + "=" * 70)
    logger.error("FAILED TESTS DETAILS")
    logger.error("=" * 70)
    for test in failed_tests:
        logger.error(f"\n❌ {test['constraint']}")
        logger.error(f"   Table: {test['table']}")
        logger.error(f"   Type: {test['type']}")
        logger.error(f"   Error: {test['error']}")
    logger.error("\n" + "=" * 70)
    logger.error(f"⚠️  {failed} TEST(S) FAILED - REVIEW REQUIRED")
    logger.error("=" * 70)

    raise Exception(f"FK/PK constraint tests failed: {failed}/{total}")
else:
    logger.info("\n" + "=" * 70)
    logger.info("🎉 ALL FK/PK CONSTRAINT METADATA TESTS PASSED!")
    logger.info("=" * 70)
    logger.info(f"Coverage: {total} constraints checked")
    logger.info("=" * 70)
